# Ingestion playground

Trigger the connectors / batch loaders on demand and see what reaches the graph: the relevance gate's keep/drop decisions, the fallback path, and the rows that land in the `signals` table.

## Ingestion architecture

```mermaid
flowchart LR
    OD["on-demand<br/>agentic-scd-collect"] --> FETCH
    POLL["scheduled poller<br/>APScheduler"] --> FETCH
    BATCH["batch loaders<br/>agentic-scd-batch<br/>Freightos · Kaggle"] --> FETCH
    WH["supplier webhook<br/>POST /signals"] --> NORM

    FETCH["fetch_with_fallback<br/>live → cached / synthetic"]
    FETCH -. raw .-> SNAP["snapshot<br/>JSON (gitignored)"]
    FETCH --> NORM["normalize →<br/>DisruptionSignal"]
    NORM --> GATE{"relevance gate<br/>Stage 0 source · Stage 1 lexicon"}
    GATE -->|dropped| REJ[("seen_rejected<br/>dedup_hash")]
    GATE -->|kept| DEDUPE["dedupe<br/>dedup_hash"]
    DEDUPE --> SIG[("signals<br/>status = 'new'")]
    SIG -->|"new → processing"| ING["ingestion_node"]
    ING --> GRAPH["GraphState.new_signals<br/>→ classify → … → recommend"]
```

**How it fits together.** Four triggers feed **one shared tail**. The on-demand
collector, scheduled poller, and batch loaders pull from their sources through
`fetch_with_fallback` (every live source degrades to a cached/synthetic fallback);
a supplier webhook pushes events straight in. Raw pulls are snapshotted to gitignored
JSON for replay. From `normalize` onward **every** trigger shares the same path: the
relevance gate (Stage 0 source targeting + Stage 1 keyword lexicon) splits keep/drop,
dropped hashes go to `seen_rejected`, kept signals are deduped and persisted to
`signals` with `status = 'new'`. The graph reads that table **out of band** —
`ingestion_node` claims the `new` rows (flipping them to `processing`) and emits them
as `new_signals` — so a busy pipeline never blocks ingestion. The cells below exercise
the middle of this diagram: `collect()`, the relevance gate, and the `signals` table.

## Run all enabled connectors once (in-process)

`collect()` runs every source in `sources.yaml` through fetch → snapshot → normalize → relevance gate → dedupe → persist. With no DB it still runs in-memory (nothing persisted) and never crashes.

In [ ]:
from agentic_scd.ingestion.collect import collect, print_summary

summary = collect()
print_summary(summary)

## Relevance gate up close

The gate keeps supply-chain-relevant signals and drops noise (Stage 0 source targeting + Stage 1 keyword lexicon). Here we mix synthetic disruptions with one off-topic item and see the split.

In [ ]:
from datetime import UTC, datetime

from agentic_scd.ingestion.connectors.synthetic import SyntheticConnector
from agentic_scd.ingestion.normalize import normalize
from agentic_scd.ingestion.relevance import gate
from agentic_scd.ingestion.schema import DisruptionSignal

connector = SyntheticConnector(name="demo", reliability=0.6, count=2)
signals = [normalize(item, connector) for item in connector.fetch()]

# Add an obviously off-topic signal so we can watch it get dropped.
signals.append(
    DisruptionSignal(
        signal_id="noise-1",
        source="demo_news",
        source_type="RSS",
        fetched_at=datetime.now(UTC),
        title="Celebrity gossip roundup",
        raw_text="No supply chain content here at all.",
    )
)

kept, dropped = gate(signals)
print("KEPT:")
for s in kept:
    print("  +", s.title)
print("DROPPED:")
for s in dropped:
    print("  -", s.title)

## Inspect the `signals` table

If Postgres is up (see the Setup section in `00_orchestration`), look at the most recent rows and their `status` (`new` → not yet read by the graph; `processing` → already drained by `ingestion_node`).

In [ ]:
import psycopg

from agentic_scd.db import DatabaseNotConfiguredError, connect

try:
    with connect() as conn, conn.cursor() as cur:
        cur.execute(
            "SELECT source, source_type, status, title "
            "FROM signals ORDER BY created_at DESC LIMIT 10"
        )
        for source, source_type, status, title in cur.fetchall():
            print(f"[{status:<10}] {source:<16} {source_type:<8} {title[:50]}")
except (DatabaseNotConfiguredError, psycopg.OperationalError) as exc:
    print(f"No DB available ({exc}). Run 00_orchestration's Setup to bring it up.")